In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path
from lifelines import CoxPHFitter
from gbmcollaboration import AllDataCenters, CHUVDataCenter
from gbmcollaboration.definitions import AVG_NUM_DAYS_PER_YEAR
from gbmcollaboration.utils import viz_survival

def load_clinical():
    def _get_clin_key(source_dict,key):
        df = (pd.concat(
            {k: v["clinical"][key] 
            for k,v in source_dict.items()})
            .reset_index()
            .rename(columns={'level_0':'center'}))
        return df

    source_dict = AllDataCenters.load_tabular(exclude_idh1_mut=True)
    df_clin_processed = _get_clin_key(source_dict,"processed gbm clinical")

    # add days unit and signed format (-1 for censored)
    df_clin_processed['os_days'] = df_clin_processed['os_years'] * AVG_NUM_DAYS_PER_YEAR
    df_clin_processed['pfs_days'] = df_clin_processed['pfs_years'] * AVG_NUM_DAYS_PER_YEAR
    df_clin_processed['os_days_signed'] = df_clin_processed['os_days'] * (df_clin_processed['os_event'].apply(lambda e: 1 if e == 1 else -1))
    df_clin_processed['pfs_days_signed'] = df_clin_processed['pfs_days'] * (df_clin_processed['pfs_event'].apply(lambda e: 1 if e == 1 else -1))

    return df_clin_processed



def plot_partial_hazard_grid(cox_model, df, continuous_cols, categorical_cols,
                             n_points=100, figsize=(5,5)):
    n_plots = len(continuous_cols) * len(categorical_cols)
    n_cols = min(3, len(categorical_cols))
    n_rows = int(np.ceil(n_plots / n_cols))
    
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(figsize[0]*n_cols, figsize[1]*n_rows))
    axs = axs.flatten()
    
    plot_idx = 0
    for cont_var in continuous_cols:
        for cat_var in categorical_cols:
            ax = axs[plot_idx]
            plot_idx += 1
            
            x_vals = np.linspace(df[cont_var].min(), df[cont_var].max(), n_points)
            strata_levels = sorted(df[cat_var].dropna().unique())
            base_row = df.median(numeric_only=True).to_dict()
            
            preds = []
            for stratum in strata_levels:
                tmp = pd.DataFrame([{**base_row, cont_var: x, cat_var: stratum} for x in x_vals])
                hazard = cox_model.predict_partial_hazard(tmp).values.flatten()
                preds.append(pd.DataFrame({
                    cont_var: x_vals,
                    cat_var: stratum,
                    'partial_hazard': hazard
                }))
            
            df_preds = pd.concat(preds, ignore_index=True)
            sns.lineplot(data=df_preds, x=cont_var, y='partial_hazard',
                         hue=cat_var, palette='tab10', ax=ax)
            
            ax.set_title(f"{cont_var} stratified by {cat_var}")
            ax.set_xlabel(cont_var.replace('_',' ').title())
            ax.set_ylabel("Predicted Partial Hazard")
            ax.grid(True, linestyle='--', alpha=0.6)
            ax.legend(title=cat_var.replace('_',' ').title())
    
    # Hide unused axes
    for i in range(plot_idx, len(axs)):
        axs[i].axis('off')
    
    plt.tight_layout()
    plt.show()


# Parameters

In [ ]:
RESULTS_DIR = Path('/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/results')
GENE_SET_SCORES_PATH = RESULTS_DIR / 'GSEA/pseudobulk_ulm_scores/ulm_scores_all.csv'
MUTATIONS_PATH = RESULTS_DIR / 'WES/mutations/patient_gene_matrix.csv'
CELL_TYPE_COLUMN = "celltype_level2_curated"
CELL_TYPE_MAPPING_CSV = '../../gbmcollaboration/metadata/SingleCell_Annotation_Key_v3.csv'

# Load data

In [ ]:
# load sc data
ad = AllDataCenters.load_singlecell(exclude_idh1_mut=True)

# load cell type name mapping
df_ct_map = pd.read_csv(CELL_TYPE_MAPPING_CSV)

# create columns with simplified cell type names
for i in range(1,5):
    col = f'celltype_level{i}_curated'
    _map = df_ct_map.set_index('celltype_level4_scanvi')[col]
    ad.obs[col] = ad.obs['celltype_level4_scanvi'].astype(str).replace(_map)

# create aggregated immune infiltrate category
ct_to_agg = ['Microglia','T_NK','Mast_cell','DC','Mono_Macro']
ad.obs[CELL_TYPE_COLUMN+'_aggregated'] = ad.obs[CELL_TYPE_COLUMN].replace({k:'immune_infiltrate' for k in ct_to_agg})
ad.obs['Subject Id'] = ad.obs['orig.ident'].str[:-1]
df_ct_props = ad.obs.groupby('Subject Id')[CELL_TYPE_COLUMN+'_aggregated'].value_counts(normalize=True).unstack(fill_value=0.)

In [ ]:
# clinical columns of interest
cols = [
    # general info
    'Site code','Subject Id','Date of cancer diagnosis',
    # survival info
    'os_days_signed','os_days','os_years','os_event','os_start_event',
    'pfs_days_signed','pfs_days','pfs_years','pfs_event','pfs_start_event',
    # predictive vars
    'Administrative gender',
    'Age (years)',
    'Performance status (ECOG) at cancer diagnosis',
    'MGMT promoter methylation'
]

# load clinical data
df_clin_processed = load_clinical()
# df_clin_processed.columns.to_frame().to_csv('1c_clinical_variables.csv',index=False)
#   remove treated by IT or ADC
df_clin_processed_no_dups = df_clin_processed[~df_clin_processed[['received_immunotherapy','received_antibody_drug_conjugate']].any(axis=1)]
#   remove duplicates
df_clin_processed_no_dups = df_clin_processed_no_dups[cols].drop_duplicates().set_index('Subject Id')

# load predictive variables
def _drop_dups(df):
    df['Subject Id'] = df.index.str[:-1]
    df_patients = df.set_index("Subject Id").loc[lambda x: ~x.index.duplicated(keep="first")]
    return df_patients

df_scores = pd.read_csv(GENE_SET_SCORES_PATH,index_col='patient_id')
df_mutations = pd.read_csv(MUTATIONS_PATH,index_col=0)
df_scores_pat = _drop_dups(df_scores)
df_mutations_pat = _drop_dups(df_mutations)


# join clinical info to scores and mutations
df = df_clin_processed_no_dups.join([df_scores_pat,df_mutations_pat]).dropna(axis=1,how='all')
#   drop constant columns
df = df.loc[:, df.nunique() > 1].copy()

non_predictive_cols = [
       'Site code', 'Date of cancer diagnosis', 'os_years', 'os_event',
       'pfs_years', 'pfs_event', 'os_days', 'pfs_days', 'os_days_signed',
       'pfs_days_signed'
]
numeric_cols = df.select_dtypes(include='number').columns.difference(non_predictive_cols)
categorical_cols = df.select_dtypes(exclude='number').columns.difference(non_predictive_cols)

# join with cell type props here not to add them to numeric cols list
df = df.join(df_ct_props)

#   print some info
common_blocks = df_scores.index.intersection(df_mutations.index)
common_patients = df_scores_pat.index.intersection(df_mutations_pat.index)
print(f'Found {len(common_blocks)} tumor blocks in common between scores ({len(df_scores)}) and mutations ({len(df_mutations)})\nSubsetting')
print(f'Found {len(common_patients)} patients in common between scores ({len(df_scores_pat)}) and mutations ({len(df_mutations_pat)})\nSubsetting')
print('Final patient level df:', df.shape)

# KM for categorical columns

In [ ]:
for x in ['os_days_signed','pfs_days_signed']:
    f, axs = plt.subplots(5,5,figsize=(30,30))
    plt.suptitle(x,y=.91,fontsize=20)
    axs = axs.flat

    for col in categorical_cols:
        ax = next(axs)

        df_plot = df.dropna(subset=col)
        add_hazard_ratio = df_plot[col].nunique() <= 2
        viz_survival.kmplot(
            df.dropna(subset=col),
            x=x,hue=col,
            add_hazard_ratio = add_hazard_ratio,
            add_logrank_pvalue = True,
            ax=ax)

# Partial HR for continuous columns

In [ ]:
strata = ['MGMT promoter methylation','Administrative gender'] 

for outcome in ['os','pfs']:
    print(outcome)

    duration_col = f'{outcome}_days'
    event_col = f'{outcome}_event'

    cox_df = df[[duration_col, event_col] + numeric_cols.tolist() + strata].dropna()
    # cox_df_encoded = pd.get_dummies(cox_df, columns=categorical_cols, drop_first=True)

    cph = CoxPHFitter()
    cph.fit(cox_df, duration_col=duration_col, event_col=event_col, strata=strata)
    # plot_partial_hazard_grid(cph, df, numeric_cols, strata, n_points=100, figsize=(5,5))
    display(cph.summary[['exp(coef)','exp(coef) lower 95%',	'exp(coef) upper 95%','p']].round(3))

# Partial HR for cell type proportions

In [ ]:
strata = ['MGMT promoter methylation','Administrative gender'] 

for outcome in ['os','pfs']:
    print(outcome)

    duration_col = f'{outcome}_days'
    event_col = f'{outcome}_event'

    cox_df = df[[duration_col, event_col] + numeric_cols.tolist()[:1] + ['immune_infiltrate'] + strata].dropna()
    # cox_df_encoded = pd.get_dummies(cox_df, columns=categorical_cols, drop_first=True)

    cph = CoxPHFitter()
    cph.fit(cox_df, duration_col=duration_col, event_col=event_col, strata=strata)
    # plot_partial_hazard_grid(cph, df, numeric_cols, strata, n_points=100, figsize=(5,5))
    display(cph.summary[['exp(coef)','exp(coef) lower 95%',	'exp(coef) upper 95%','p']].round(3))